In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objs as go
from statsmodels.tsa.seasonal import seasonal_decompose
from template import API  # Assumes template.API.py is in your Python path

In [ ]:
df = API.fetch_bitcoin_metric("transaction_count")
df.head()

In [ ]:
# Fill missing values
df["value"].interpolate(method="linear", inplace=True)

# Rolling calculations
df["rolling_mean"] = df["value"].rolling(window=10, min_periods=1).mean()
df["rolling_std"] = df["value"].rolling(window=10, min_periods=1).std()
df["z_score"] = (df["value"] - df["rolling_mean"]) / df["rolling_std"]

In [ ]:
decomposition = seasonal_decompose(df["value"], model="additive", period=10)
df["trend"] = decomposition.trend
df["seasonal"] = decomposition.seasonal
df["residual"] = decomposition.resid

In [ ]:
fig = go.Figure()

# Actual values
fig.add_trace(go.Scatter(x=df.index, y=df["value"], name="Transaction Count", mode="lines"))

# Rolling mean
fig.add_trace(go.Scatter(x=df.index, y=df["rolling_mean"], name="Rolling Mean", mode="lines"))

# Anomalies
anomalies = df[df["z_score"].abs() > 2]
fig.add_trace(go.Scatter(x=anomalies.index, y=anomalies["value"],
                         mode="markers", name="Anomalies", marker=dict(color="red", size=8)))

fig.update_layout(title="Bitcoin Transaction Count + Anomalies", xaxis_title="Date", yaxis_title="Count")
fig.show()

In [ ]:
import plotly.io as pio
pio.write_html(fig, file="bitcoin_transaction_plot.html", auto_open=False)